# The condition of the monomial basis

The columns $1, x, x^2, \ldots, x^d$ for $x = 1, 2, \ldots, n$ and a large $d$ are mathematically independent and numerically almost dependent. Mapping the inputs onto $[-1, 1]$ before the powers are formed cuts the condition number by orders of magnitude. The fitted polynomial, in exact arithmetic, does not change.


## What exact arithmetic says

Let $x = 1, 2, \ldots, 10$ and $y = 1 + 2x$. Consider a polynomial of degree $8$. The coefficient vector

$$
(1,\ 2,\ 0,\ 0,\ 0,\ 0,\ 0,\ 0,\ 0)
$$

has residual sum of squares zero. The Vandermonde matrix on ten distinct inputs with nine columns has full column rank, because a nonzero polynomial of degree at most $8$ cannot vanish at ten points. Therefore this coefficient vector is the unique least-squares solution. Every coefficient of $x^2, \ldots, x^8$ is exactly zero, and the intercept and slope are exactly $1$ and $2$.

The same statement survives a linear change of coordinate. Set

$$
u = \dfrac{2(x - 1)}{9} - 1,
$$

so $x = 1$ becomes $u = -1$ and $x = 10$ becomes $u = 1$. Then $x$ is linear in $u$, and $y = 1 + 2x$ is linear in $u$. In the basis $1, u, \ldots, u^8$ the coefficients of $u^2, \ldots, u^8$ are exactly zero as well.


In [1]:
# The monomial vector (1, 2, 0, ..., 0) fits y = 1 + 2x with residual zero.
x = list(range(1, 11))
y = [1 + 2 * xi for xi in x]
residual = [yi - (1 + 2 * xi) for xi, yi in zip(x, y)]
print("residual sum of squares", sum(r ** 2 for r in residual))
print("distinct inputs", len(set(x)))
assert sum(r ** 2 for r in residual) == 0
assert len(set(x)) == 10


residual sum of squares 0
distinct inputs 10


## What floating-point arithmetic does

Forming $X^{\mathsf T}X$ from the raw powers $x^8$ mixes numbers of very different sizes. For this design the condition number of $X$ is about $4.01\times 10^{10}$. Solving the normal equations in 64-bit floating point does not return the exact coefficient vector. The coefficient of $x^2$, which the algebra says is zero, comes back near $-7.3\times 10^{-7}$.

The scaled inputs $u \in [-1, 1]$ have a design matrix whose condition number is about $1.04\times 10^{3}$. The same solve puts every coefficient of $u^2, \ldots, u^8$ below $3\times 10^{-10}$ in absolute value. Both fits are trying to represent the same line. The scaled basis stays closer to the exact zeros because its columns are closer to orthogonal on the scale floating-point arithmetic can see.

This is not a license to ignore centering when the degree is three and the inputs are already small. It is the reason a high degree on raw inputs such as years, or on measurements far from zero, should be rewritten in a centered and scaled coordinate before the powers are built.


In [2]:
# Float64 normal equations: raw powers drift farther from the exact zeros.
import numpy as np
x = np.arange(1, 11, dtype=float)
y = 1 + 2 * x
degree = 8
raw = np.column_stack([x ** k for k in range(degree + 1)])
beta_raw = np.linalg.solve(raw.T @ raw, raw.T @ y)
u = 2 * (x - 1) / 9 - 1
scaled = np.column_stack([u ** k for k in range(degree + 1)])
beta_scaled = np.linalg.solve(scaled.T @ scaled, scaled.T @ y)
print("raw condition", np.linalg.cond(raw))
print("scaled condition", np.linalg.cond(scaled))
print("raw x^2 coefficient", beta_raw[2])
print("largest scaled coefficient above degree 1", np.max(np.abs(beta_scaled[2:])))
assert np.linalg.cond(raw) > 1e10
assert np.linalg.cond(scaled) < 2e3
assert abs(beta_raw[2]) > 1e-7
assert np.max(np.abs(beta_scaled[2:])) < 1e-8


raw condition 40112746278.02819
scaled condition 1043.5843248084673
raw x^2 coefficient -7.282845161427841e-07
largest scaled coefficient above degree 1 2.5261068299976475e-10


## Pitfall

A large condition number does not mean the polynomial model is unidentified. In exact arithmetic the coefficient of $x^2$ is zero and the fit is unique. The warning belongs to the arithmetic, and to the variance of the coefficients when the columns are nearly dependent. Scaling $x$ into $[-1, 1]$ changes the reported coefficients, as the shift of origin already did, and it leaves the fitted values unchanged in exact arithmetic. Judging a scaled coefficient against a raw coefficient, as if they multiplied the same column, repeats the mistake of comparing $2$ with $-58$.


In [3]:
# Scaled inputs run from -1 to 1. They are a coordinate change, not a new response.
import numpy as np
x = np.arange(1, 11, dtype=float)
u = 2 * (x - 1) / 9 - 1
print("scaled endpoints", u[0], u[-1])
assert abs(u[0] + 1) < 1e-12 and abs(u[-1] - 1) < 1e-12


scaled endpoints -1.0 1.0


## Summary

- For $y = 1 + 2x$ on $x = 1,\ldots,10$, every least-squares coefficient of degree $2$ through $8$ is exactly zero.
- In 64-bit arithmetic the raw degree-$8$ design has condition number about $4\times 10^{10}$ and a spurious $x^2$ coefficient near $-7\times 10^{-7}$.
- Mapping $x$ onto $[-1, 1]$ drops the condition number to about $10^3$ and keeps the higher coefficients under $10^{-8}$.
